# 08 — Robustness Battery

Test whether the M3a within-between findings hold under the robustness checks implemented in this repository:

| # | Check | Notes |
| --- | --- | --- |
| 2 | M3a vs M3b (round-dummy specification) | already in notebook 07; carried forward here |
| 4 | Leave-one-out exposure | per-individual `exposure_ct_loo` |
| 5 | Item-by-item outcome (`trstprl`, `trstlgl`, `stfdem`) | re-fit M3a on each item separately |
| 6 | Country leverage (drop-one) | re-fit M3a leaving each country out |
| 7 | Within-only FE specification | `linearmodels.PanelOLS`, expect $\hat\beta_{FE}\approx\hat\gamma_W$ |
| 12 | Round subsetting (drop R10, COVID-disrupted) | re-fit M3a |

Not implemented in this repository because they require additional source data or an R bridge:
1. Webb / Felten / Eloundou alternative exposures (need SOC↔ISCO crosswalk + downloads)
3. ESS-internal vs Eurostat aggregation (Eurostat LFS shares download)
8. Hausman-Mundlak (different test variant of #2)
9, 10. `pymer4` cross-validation + survey weights (require R + lme4 install)
11. MICE for `hinctnta`

Vintage-static check (using `genai_i_static` instead of vintage-applied `genai_i`) is the most informative one we *can* do — it isolates whether the within-effect comes from the R10→R11 vintage shift or from compositional drift.

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.models import (  # noqa: E402
    add_country_year_key,
    build_trust_composite,
    fit_3level,
    mundlak_wald,
)
from src.mla.mundlak import (  # noqa: E402
    within_between_decompose,
)

ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
INTERIM_DIR  = REPO_ROOT / "data" / "interim"
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

In [2]:
def recode_sentinels(s, lo, hi):
    return s.where(s.between(lo, hi))

df = pd.read_parquet(ANALYSIS_DIR / "analysis.parquet")
df = build_trust_composite(df)
df = add_country_year_key(df)
df["agea"]    = recode_sentinels(df["agea"], 14, 110)
df["gndr"]    = recode_sentinels(df["gndr"], 1, 2)
df["eisced"]  = recode_sentinels(df["eisced"], 0, 7)
df["hinctnta"] = recode_sentinels(df["hinctnta"], 1, 10)
df["mnactic"] = recode_sentinels(df["mnactic"], 1, 9)
df["domicil"] = recode_sentinels(df["domicil"], 1, 5)
df["agea_c"] = df["agea"] - 45
df["agea_c_sq"] = df["agea_c"] ** 2
df["female"] = (df["gndr"] == 2).astype("float64")
for _c in ("essround", "isco08", "year"):
    if str(df[_c].dtype).startswith("Int"):
        df[_c] = df[_c].astype("float64")
df["genai_z"] = (df["genai_i"] - df["genai_i"].mean()) / df["genai_i"].std()

REQUIRED = [
    "trust", "genai_i", "genai_z", "eisced", "agea_c", "female", "mnactic",
    "domicil", "hinctnta", "gdp_growth", "unemp_rate", "hicp_inflation",
    "exposure_ct", "exposure_ct_within", "exposure_ct_between",
]
df_fit = df.dropna(subset=REQUIRED).copy()
M3A_FORMULA = (
    "trust ~ genai_z + C(eisced) + agea_c + agea_c_sq + female "
    "+ C(mnactic) + C(domicil) + hinctnta "
    "+ gdp_growth + unemp_rate + hicp_inflation + C(essround) "
    "+ exposure_ct_within + exposure_ct_between"
)
print(f"common-sample N: {len(df_fit):,}, {df_fit.cntry.nunique()} countries")

common-sample N: 165,969, 30 countries


## R5: Item-by-item outcome (`trstprl`, `trstlgl`, `stfdem` separately)

Re-fit M3a using each trust item alone (z-standardised globally) instead of the composite, to check whether one item dominates the result.

In [3]:
def fit_for_item(item: str):
    item_df = df_fit.copy()
    s = item_df[item].where(item_df[item].between(0, 10))
    item_df["trust_item"] = (s - s.mean()) / s.std()
    item_df = item_df.dropna(subset=["trust_item"])
    formula = M3A_FORMULA.replace("trust ~", "trust_item ~")
    res = fit_3level(formula, item_df)
    test = mundlak_wald(res, "exposure_ct_within", "exposure_ct_between")
    return {
        "item": item,
        "n": int(res.nobs),
        "gamma_w": test["gamma_w"],
        "se_w": res.bse["exposure_ct_within"],
        "gamma_b": test["gamma_b"],
        "se_b": res.bse["exposure_ct_between"],
        "wald_chi2": test["chi2"],
        "wald_p": test["pvalue"],
    }

by_item = pd.DataFrame([fit_for_item(c) for c in ("trstprl", "trstlgl", "stfdem")])
for c in by_item.select_dtypes("float64").columns:
    by_item[c] = by_item[c].round(4)
by_item

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,item,n,gamma_w,se_w,gamma_b,se_b,wald_chi2,wald_p
0,trstprl,164064,1.3826,1.3232,11.2629,3.7959,6.0332,0.0140
1,trstlgl,163979,-1.8302,1.2862,12.0299,4.0265,10.7409,0.0010
2,stfdem,162735,0.5688,1.4432,11.0145,3.5233,7.5135,0.0061


## R6: Country leverage (drop-one)

Re-fit M3a leaving each country out in turn. Distribution of γ_W and γ_B across drops shows whether any single country is driving the result.

In [4]:
drop_records = []
for c in sorted(df_fit["cntry"].unique()):
    sub = df_fit[df_fit["cntry"] != c]
    res = fit_3level(M3A_FORMULA, sub)
    test = mundlak_wald(res, "exposure_ct_within", "exposure_ct_between")
    drop_records.append({
        "dropped": c,
        "n_left": int(res.nobs),
        "gamma_w": test["gamma_w"],
        "gamma_b": test["gamma_b"],
        "wald_p": test["pvalue"],
    })
drops = pd.DataFrame(drop_records).sort_values("gamma_b")
for c in drops.select_dtypes("float64").columns:
    drops[c] = drops[c].round(3)
print("Range across drops:")
print(f"  γ_W: [{drops['gamma_w'].min()}, {drops['gamma_w'].max()}]")
print(f"  γ_B: [{drops['gamma_b'].min()}, {drops['gamma_b'].max()}]")
print(f"  p:   [{drops['wald_p'].min()}, {drops['wald_p'].max()}]")
print()
print('Most influential drops (top 5 by |Δγ_B|):')
baseline_b = mundlak_wald(
    fit_3level(M3A_FORMULA, df_fit), 'exposure_ct_within', 'exposure_ct_between'
)['gamma_b']
drops['delta_b'] = (drops['gamma_b'] - baseline_b).abs()
drops.sort_values('delta_b', ascending=False).head(5)

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


...

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Range across drops:
  γ_W: [-0.348, 0.358]
  γ_B: [9.459, 12.243]
  p:   [0.001, 0.012]

Most influential drops (top 5 by |Δγ_B|):


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


,dropped,n_left,gamma_w,gamma_b,wald_p,delta_b
2,BG,159056,0.194,9.459,0.012,1.779962
3,CH,159924,0.143,10.076,0.008,1.162962
4,CY,164245,-0.096,12.243,0.001,1.004038
20,ME,165544,0.097,12.209,0.001,0.970038
5,CZ,158865,-0.273,12.202,0.001,0.963038


## R7: Within-only FE specification (linearmodels.PanelOLS)

Country fixed effects + cluster-robust SEs. The country-FE coefficient on `exposure_ct` should approximately equal $\hat\gamma_W$ from M3a (Mundlak identity).

In [5]:
from linearmodels.panel import PanelOLS

# Build the regressor frame BEFORE setting the panel index, then attach
# the (cntry, essround) MultiIndex at the end.
fe_X = df_fit[[
    "genai_z", "agea_c", "agea_c_sq", "female", "hinctnta",
    "gdp_growth", "unemp_rate", "hicp_inflation",
    "exposure_ct",
]].copy()
fe_X = pd.concat([
    fe_X,
    pd.get_dummies(df_fit["eisced"].astype(int), prefix="eisced", drop_first=True).astype(float),
    pd.get_dummies(df_fit["essround"].astype(int), prefix="essround", drop_first=True).astype(float),
], axis=1)
y = df_fit["trust"].copy()

# linearmodels needs a 2-level index: (entity, time). Use cntry × essround.
idx = pd.MultiIndex.from_arrays([df_fit["cntry"], df_fit["essround"]], names=["cntry", "essround"])
fe_X.index = idx
y.index = idx

fe = PanelOLS(y, fe_X, entity_effects=True, drop_absorbed=True).fit(
    cov_type="clustered", cluster_entity=True
)
fe_b = float(fe.params.get("exposure_ct", float("nan")))
fe_se = float(fe.std_errors.get("exposure_ct", float("nan")))
print(f"FE β on exposure_ct: {fe_b:+.4f}  cluster-robust SE: {fe_se:.4f}")
print("M3a γ_W (within) for comparison: see notebook 07.")

FE β on exposure_ct: -0.1887  cluster-robust SE: 0.7872
M3a γ_W (within) for comparison: see notebook 07.


## R12: Drop R10 (COVID-disrupted, mode-mixed fieldwork)

In [6]:
df_no_r10 = df_fit[df_fit["essround"] != 10].copy()
print(f"sample without R10: {len(df_no_r10):,}, {df_no_r10.cntry.nunique()} countries")
res = fit_3level(M3A_FORMULA, df_no_r10)
test = mundlak_wald(res, "exposure_ct_within", "exposure_ct_between")
print(f"γ_W = {test['gamma_w']:+.4f} (SE {res.bse['exposure_ct_within']:.4f})")
print(f"γ_B = {test['gamma_b']:+.4f} (SE {res.bse['exposure_ct_between']:.4f})")
print(f"Wald χ²(1) = {test['chi2']:.2f}, p = {test['pvalue']:.4f}")

sample without R10: 143,451, 29 countries


γ_W = -0.8695 (SE 1.2354)
γ_B = +10.5166 (SE 3.3324)
Wald χ²(1) = 10.26, p = 0.0014


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## R-V: Vintage-static — re-build the within-between using `genai_i_static`

If the result we see in M3a comes mainly from the R10→R11 vintage shock, fitting on the static-vintage exposure (2025 throughout) should weaken or extinguish the within-effect.

In [7]:
# We already persisted exposure_ct_static_within / exposure_ct_static_between in notebook 03.
static_formula = M3A_FORMULA.replace(
    "exposure_ct_within + exposure_ct_between",
    "exposure_ct_static_within + exposure_ct_static_between",
)
df_s = df_fit.dropna(subset=["exposure_ct_static_within", "exposure_ct_static_between"]).copy()
res_static = fit_3level(static_formula, df_s)
test_s = mundlak_wald(res_static, "exposure_ct_static_within", "exposure_ct_static_between")
print(f"static-vintage γ_W = {test_s['gamma_w']:+.4f} (SE {res_static.bse['exposure_ct_static_within']:.4f})")
print(f"static-vintage γ_B = {test_s['gamma_b']:+.4f} (SE {res_static.bse['exposure_ct_static_between']:.4f})")
print(f"Wald χ²(1) = {test_s['chi2']:.2f}, p = {test_s['pvalue']:.4f}")

static-vintage γ_W = -0.2357 (SE 1.2792)
static-vintage γ_B = +13.4271 (SE 3.5748)
Wald χ²(1) = 13.00, p = 0.0003


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## R4: Leave-one-out exposure

In [8]:
# Re-build the country-year exposure using the leave-one-out per-row mean,
# then re-derive within / between for that variable.
df_loo = df_fit.dropna(subset=["exposure_ct_loo"]).copy()
loo_cy = (
    df_loo.groupby(["cntry", "essround"], observed=True)["exposure_ct_loo"]
    .mean()
    .rename("exposure_ct_loo_avg")
    .reset_index()
)
loo_cy = within_between_decompose(loo_cy, "exposure_ct_loo_avg")
df_loo = df_loo.drop(columns=[c for c in ("exposure_ct_loo_avg", "exposure_ct_loo_avg_within", "exposure_ct_loo_avg_between") if c in df_loo.columns])
df_loo = df_loo.merge(loo_cy, on=["cntry", "essround"], how="left")
loo_formula = M3A_FORMULA.replace(
    "exposure_ct_within + exposure_ct_between",
    "exposure_ct_loo_avg_within + exposure_ct_loo_avg_between",
)
res_loo = fit_3level(loo_formula, df_loo)
test_loo = mundlak_wald(res_loo, "exposure_ct_loo_avg_within", "exposure_ct_loo_avg_between")
print(f"leave-one-out γ_W = {test_loo['gamma_w']:+.4f} (SE {res_loo.bse['exposure_ct_loo_avg_within']:.4f})")
print(f"leave-one-out γ_B = {test_loo['gamma_b']:+.4f} (SE {res_loo.bse['exposure_ct_loo_avg_between']:.4f})")
print(f"Wald χ²(1) = {test_loo['chi2']:.2f}, p = {test_loo['pvalue']:.4f}")

leave-one-out γ_W = +0.0633 (SE 1.1256)
leave-one-out γ_B = +10.4259 (SE 3.3776)
Wald χ²(1) = 8.49, p = 0.0036


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## Robustness summary table

In [9]:
summary_rows = []
summary_rows.append({
    "check": "R5 trstprl alone", **{k: by_item[by_item['item']=='trstprl'][k].iloc[0] for k in ('gamma_w','gamma_b','wald_p')},
})
summary_rows.append({
    "check": "R5 trstlgl alone", **{k: by_item[by_item['item']=='trstlgl'][k].iloc[0] for k in ('gamma_w','gamma_b','wald_p')},
})
summary_rows.append({
    "check": "R5 stfdem alone",  **{k: by_item[by_item['item']=='stfdem'][k].iloc[0] for k in ('gamma_w','gamma_b','wald_p')},
})
summary_rows.append({
    "check": "R6 country drop range (γ_B min/max)",
    "gamma_w": f"[{drops['gamma_w'].min()}, {drops['gamma_w'].max()}]",
    "gamma_b": f"[{drops['gamma_b'].min()}, {drops['gamma_b'].max()}]",
    "wald_p": f"[{drops['wald_p'].min()}, {drops['wald_p'].max()}]",
})
summary_rows.append({
    "check": "R12 drop R10", "gamma_w": round(test['gamma_w'], 4), "gamma_b": round(test['gamma_b'], 4), "wald_p": round(test['pvalue'], 4),
})
summary_rows.append({
    "check": "R-V vintage-static", "gamma_w": round(test_s['gamma_w'], 4), "gamma_b": round(test_s['gamma_b'], 4), "wald_p": round(test_s['pvalue'], 4),
})
summary_rows.append({
    "check": "R4 leave-one-out", "gamma_w": round(test_loo['gamma_w'], 4), "gamma_b": round(test_loo['gamma_b'], 4), "wald_p": round(test_loo['pvalue'], 4),
})
summary_rows.append({
    "check": "R7 PanelOLS β on exposure_ct (within-only FE)", "gamma_w": round(fe_b, 4), "gamma_b": "n/a", "wald_p": "n/a",
})
robust = pd.DataFrame(summary_rows)
robust

,check,gamma_w,gamma_b,wald_p
0,R5 trstprl alone,1.3826,11.2629,0.014
1,R5 trstlgl alone,-1.8302,12.0299,0.001
2,R5 stfdem alone,0.5688,11.0145,0.0061
3,R6 country drop range (γ_B min/max),"[-0.348, 0.358]","[9.459, 12.243]","[0.001, 0.012]"
4,R12 drop R10,-0.8695,10.5166,0.0014
5,R-V vintage-static,-0.2357,13.4271,0.0003
6,R4 leave-one-out,0.0633,10.4259,0.0036
7,R7 PanelOLS β on exposure_ct (within-only FE),-0.1887,n/a,n/a


In [10]:
# Persist for the paper. The summary table mixes floats and range
# strings, so write it as CSV (parquet wants column-uniform dtypes).
robust.to_csv(INTERIM_DIR / "robustness_summary.csv", index=False)
by_item.to_parquet(INTERIM_DIR / "robustness_item_by_item.parquet", index=False)
drops.to_parquet(INTERIM_DIR / "robustness_country_drop.parquet", index=False)
print("persisted: robustness_summary (csv), robustness_item_by_item, robustness_country_drop")

persisted: robustness_summary (csv), robustness_item_by_item, robustness_country_drop
